# Cart-pole PPO training in Colab
MuJoCo physics + Gymnasium environment + Stable Baselines3 PPO. This notebook uses the same source-explicit observations and raw acceleration contract as the SDK. No rig connection is made.

Prepare `cartpole-source.zip` from the checkout using `python -m zipfile -c cartpole-source.zip pyproject.toml README.md uv.lock src examples scripts`. Upload it with reviewed `model.json`, `executor-parameters.json`, `observation-config.json`, and `prior-parameters.json`. Read `docs/rl-training.md` first. Current model scopes require explicit DEVELOPMENT_ONLY simulation; rejected fits are never accepted and no hardware qualification is claimed.

In [ ]:
from google.colab import files
uploaded = files.upload()  # source archive and the four explicit input JSON files


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile
REPO = Path('/content/cartpole_training')
REPO.mkdir(exist_ok=True)
with zipfile.ZipFile('/content/cartpole-source.zip') as archive:
    for member in archive.infolist():
        if not (REPO / member.filename).resolve().is_relative_to(REPO.resolve()):
            raise ValueError('Archive member outside source directory')
    archive.extractall(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'torch>=2.8,<3', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO}[rl]'], check=True)
print('CPU training dependencies ready. Actual versions are saved in each checkpoint manifest.')


In [ ]:
import json
INPUTS = Path('/content')
names = ['model.json', 'executor-parameters.json', 'observation-config.json', 'prior-parameters.json']
for name in names:
    if not (INPUTS / name).is_file():
        raise FileNotFoundError(f'Upload reviewed {name}')
model_document = json.loads((INPUTS / 'model.json').read_text())
print('Model schema:', model_document.get('schema'))
print('Model scope:', model_document.get('acceptance_scope', model_document.get('status')))
print('Model pin:', model_document.get('content_sha256'))
print('No global swing-up or hardware qualification is inferred from this scope.')


## Train a small execution check, then choose a larger budget
The default below is 2,048 steps, not a performance experiment. DEVELOPMENT_ONLY is an explicit exploratory-simulation acknowledgement, not a bypass for rejected fits. Preserve a final held-out seed set; do not tune against it. Each output directory must be new. Keep the source/archive and inputs so a Colab reset does not lose them.

In [ ]:
DEVELOPMENT_ONLY = True  # Current scopes do not qualify global swing-up or hardware.
STEPS = 2048  # Must be a multiple of N_STEPS; increase only after reviewing model adequacy.
N_STEPS, BATCH_SIZE, EPOCHS = 1024, 64, 10
RUN = Path('/content/ppo-run-001')
def train_command(output, steps, resume=None):
    command = [sys.executable, str(REPO / 'examples/train_ppo.py'), 'train',
        '--model', str(INPUTS / 'model.json'),
        '--executor-parameters', str(INPUTS / 'executor-parameters.json'),
        '--observation-config', str(INPUTS / 'observation-config.json'),
        '--prior-parameters', str(INPUTS / 'prior-parameters.json'),
        '--training-seeds', '11', '12', '13', '--seed', '20261017', '--seconds', '10',
        '--steps', str(steps), '--n-steps', str(N_STEPS), '--batch-size', str(BATCH_SIZE),
        '--epochs', str(EPOCHS), '--checkpoint-every', str(N_STEPS), '--output', str(output)]
    if DEVELOPMENT_ONLY: command.append('--development-only')
    if resume is not None: command += ['--resume', str(resume)]
    return command
subprocess.run(train_command(RUN, STEPS), check=True)
print(json.loads((RUN / 'manifest.json').read_text()))


In [ ]:
DO_RESUME = False
if DO_RESUME:
    resumed = Path('/content/ppo-run-002')
    subprocess.run(train_command(resumed, STEPS, resume=RUN), check=True)
    RUN = resumed
# Resume validates model/configuration/seeds/code/dependency versions.
# Optimizer state continues; episodes restart, so this is not bitwise uninterrupted training.


In [ ]:
REPORT = Path('/content/ppo-heldout-001.json')
subprocess.run([sys.executable, str(REPO / 'examples/train_ppo.py'), 'evaluate', str(RUN),
    '--seeds', '1001', '1002', '1003', '1004', '1005', '1006', '--output', str(REPORT)], check=True)
report = json.loads(REPORT.read_text())
print({key: report[key] for key in ('episodes_count', 'successes', 'mean_score', 'status', 'hardware_accepted')})
for episode in report['episodes']:
    print(episode)


In [ ]:
import shutil
bundle = Path('/content/ppo-result-bundle')
bundle.mkdir(exist_ok=False)
shutil.copytree(RUN, bundle / 'run')
shutil.copy2(REPORT, bundle / REPORT.name)
for name in names: shutil.copy2(INPUTS / name, bundle / name)
shutil.copy2('/content/cartpole-source.zip', bundle / 'cartpole-source.zip')
archive = shutil.make_archive('/content/ppo-results', 'zip', bundle)
files.download(archive)
# checkpoint.zip is for trusted offline resume. actor.json is numeric SDK inference data.
# Neither artifact authorizes a hardware trial.
